In [55]:
import os
from openai import OpenAI
import pandas as pd
import numpy as np
import re
from pypinyin import lazy_pinyin
from rapidfuzz import fuzz
import math
from uuid import uuid4 as uuid
from dotenv import load_dotenv
import subprocess
from tqdm import tqdm
import json
load_dotenv(".env")

root = "/mnt/NextcloudSacmData/sacm.av/files/Recordings"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
df = pd.read_csv("songs.csv")
df.tail(5)

,code,type,title,lyrics,pinyin
492,UNK-7,PnW,我安然居住,我来到祢宝座 献上我的祷告\n在这宁静时刻 我听见祢的声音\n因为祢的爱 已经找到了我\n使...,wo lai dao mi bao zuo xian shang wo de dao gao...
493,UNK-8,PnW,主我愿像祢,主我愿像祢，荣耀之救主！\n此乃我祷告亦是盼望，\n我甘愿舍弃，世上之财宝，\n只要能披戴我...,zhu wo yuan xiang mi rong yao zhi jiu zhu ci n...
494,UNK-9,PnW,谁能使我与神的爱隔绝,哦~ 神伟大的爱\n何其长阔高深\n竟不吝惜祂独生爱子\n为我们众人舍了\n神称为义的人谁能...,o ~ shen wei da de ai he qi zhang kuo gao shen...
495,UNK-10,Hymn,主活在我心,从前在罪中远离神，\n心怀毫无亮光。\n从主言语中才觉悟，\n基督活在心中。\n\n主活在我...,cong qian zai zui zhong yuan li shen xin huai ...
496,UNK-11,PnW,盼望圣灵,将我眼目 聚焦于祢 喔耶稣\n将我全心 来回应祢的爱\n放下恐惧 我全人降服于祢\n我王 我...,jiang wo yan mu ju jiao yu mi o ye su jiang wo...


In [ ]:
files = sorted([f"{d}/{f}" for d in os.listdir(root) for f in os.listdir(f"{root}/{d}")])

to_rename = {}
for i, row in df.iterrows():
    title = row.title
    title_ = re.sub(r"[，。！？、“”：；\n]", "", row.title).replace("赞美诗24", "")
    if title_ == row.title:
        continue
    df.at[i, "title"] = title_
    if not (affected := [f for f in files if title in f]):
        continue
    print(f">>>>>>>>>> {row.name}: {row.title}")
    print("\n".join(affected))
    for f in affected:
        new_name = to_rename.get(f, f).replace(title, title_)
        to_rename[f] = new_name

In [ ]:
folders = set()
for old_path, new_path in to_rename.items():
    d, _ = old_path.split("/", 1)
    folders.add(d)
    os.rename(f"{root}/{old_path}", f"{root}/{new_path}")

# for d in folders:
#     !sudo -u www-data php /var/www/html/nextcloud_sacm/occ files:scan --path sacm.av/files/Recordings/{d}

In [10]:
def pinyin(text):
    text = re.sub(r"[，。！？、“”：；\n]", " ", text)
    result = " ".join(lazy_pinyin(text))
    return re.sub(r"\s+", " ", result).strip()


def windows(tokens, size, step):
    if len(tokens) <= size:
        yield " ".join(tokens)
    else:
        for i in range(0, len(tokens) - size + 1, step):
            yield " ".join(tokens[i:i+size])


def best_window_score(query_py, lyrics_py, size=50, step=10):
    query_tokens = query_py.split()
    lyric_tokens = lyrics_py.split()
    score = max(
        fuzz.ratio(qw, lw)
        for qw in windows(query_tokens, size, step)
        for lw in windows(lyric_tokens, size, step)
    )
    return score / 100


def get_duration(filepath) -> float:
    result = subprocess.run(
        [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            filepath,
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(f"[ERROR] ffprobe failed: {result.stderr}")
        return 0
    return float(result.stdout.strip())


def split_to_limit(filepath, limit=26_214_400, margin=0.90, out_dir="tmp"):
    size = os.path.getsize(filepath)
    if size <= limit:
        return [filepath]
    os.makedirs(out_dir, exist_ok=True)
    duration = get_duration(filepath)
    bitrate_kbps = 128
    chunk_seconds = max(1, int(limit * margin * 8 / (bitrate_kbps * 1000)))
    chunk_paths = []
    
    for start in range(0, math.ceil(duration), chunk_seconds):
        chunk_path = os.path.join(out_dir, f"{uuid()}.mp3")
        subprocess.run([
            "ffmpeg",
            "-y",
            "-ss", str(start),
            "-t", str(chunk_seconds),
            "-i", filepath,
            "-vn",
            "-c:a", "libmp3lame",
            "-b:a", f"{bitrate_kbps}k",
            chunk_path,
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        chunk_paths.append(chunk_path)
        print(
            f"chunk {len(chunk_paths)}: "
            f"{os.path.getsize(chunk_path):,} bytes "
            f"(limit {limit:,}) -> {chunk_path}"
        )
    return chunk_paths

In [ ]:
def match_zoom_to_sq(d, tol=10, verbose=False):
    files = sorted(os.listdir(f"{root}/{d}"))
    zoom_files = [f for f in files if f.startswith("ZOOM")]
    sq_files = [f for f in files if not f.startswith("ZOOM")]
    
    if not zoom_files or not sq_files:
        return {}

    durations = {f: int(get_duration(f"{root}/{d}/{f}")) for f in files}
    if verbose:
        print(json.dumps(durations, indent=2, ensure_ascii=False))

    reduced_sq_files = []
    for f in sq_files:
        other_files = [x for x in reduced_sq_files if not x.startswith("SQ")]
        if f.startswith("SQ") or not any(durations[f] == durations[x] for x in other_files):
            reduced_sq_files.append(f)
    sq_files = reduced_sq_files

    if verbose:
        print(zoom_files)
        print(sq_files)

    # If same number of files, just pair in order
    if len(zoom_files) == len(sq_files):
        ordered_matches = dict(zip(zoom_files, sq_files))
        # Make sure the sizes match though
        if all(abs(durations[z] - durations[s]) < tol for z, s in ordered_matches.items()):
            return ordered_matches

    # Otherwise, take the largest filesize (P&W) pair as reference
    matches = {}

    def traverse(ref_zoom_idx, ref_sq_idx, zoom_list, sq_list):
        z_idx = ref_zoom_idx + 1
        s_idx = ref_sq_idx + 1
        while z_idx < len(zoom_list) and s_idx < len(sq_list):
            zoom_file = zoom_list[z_idx]
            for i, sq_file in enumerate(sq_list):
                if sq_file.startswith("SQ") and i < s_idx:
                    continue
                if verbose:
                    print(f"{i=} {s_idx=} {zoom_file} ({durations[zoom_file]}) : {sq_file} ({durations[sq_file]})")
                if abs(durations[zoom_file] - durations[sq_file]) > tol:
                    continue
                matches[zoom_file] = sq_file
                if sq_file.startswith("SQ"):
                    s_idx = i + 1
                break
            z_idx += 1

    def prune_same_values(d):
        counts = {v: len([k for k in d if d[k] == v]) for v in d.values()}
        return {k: v for k, v in d.items() if counts[v] == 1}

    largest_zoom_file = max(zoom_files, key=lambda f: durations[f])
    largest_sq_file = max(sq_files, key=lambda f: durations[f])
    if not largest_sq_file.startswith("SQ"):
        # since order can't be gleaned from filename, just traverse the whole list
        traverse(-1, -1, zoom_files, sq_files)
        return prune_same_values(matches)

    if abs(durations[largest_zoom_file] - durations[largest_sq_file]) <= tol:
        matches[largest_zoom_file] = largest_sq_file
        zoom_idx = zoom_files.index(largest_zoom_file)
        sq_idx = sq_files.index(largest_sq_file)
    else:
        cost = np.array([
            [abs(durations[z] - durations[s]) for s in sq_files]
            for z in zoom_files
        ])
        zoom_idx, sq_idx = np.unravel_index(np.argmin(cost), cost.shape)
        matches[zoom_files[zoom_idx]] = sq_files[sq_idx]

    traverse(zoom_idx, sq_idx, zoom_files, sq_files)
    traverse(len(zoom_files) - zoom_idx - 1, len(sq_files) - sq_idx - 1, zoom_files[::-1], sq_files[::-1])

    return prune_same_values(matches)

# matches = match_zoom_to_sq("2026-04-11")
# print(json.dumps(matches, indent=2, ensure_ascii=False))

In [56]:
lyrics = """神的子民合一聚集，
热切期盼遇见你。
天要敞开，神的道传遍，
因为我们渴慕寻求你。

在这里，在这里，耶稣你现在在这里。
我相信，我相信，喜乐的江河在这里。
荣耀彰显，奇妙神迹将显明，
天上国度降临在这里。

你的灵在这里，这里就有自由，
圣灵自由运行无止尽。
移山倒海信心，坚定不断升起，
期待伟大神做奇妙事。"""
num = max([int(code.split("-")[1]) for code in df.loc[df.code.str.startswith("UNK")].code])
df.loc[len(df)] = {
    "code": f"UNK-{num + 1}",
    "type": "PnW",
    "title": "在这里",
    "lyrics": lyrics,
    "pinyin": pinyin(lyrics),
}
# idx = 495
# df.loc[idx, "lyrics"] = lyrics
# df.loc[idx, "pinyin"] = pinyin(lyrics)
df.to_csv("songs.csv", index=False)
df.tail(1)

,code,type,title,lyrics,pinyin
497,UNK-12,PnW,在这里,神的子民合一聚集，\n热切期盼遇见你。\n天要敞开，神的道传遍，\n因为我们渴慕寻求你。\n...,shen de zi min he yi ju ji re qie qi pan yu ji...


In [ ]:
for d in sorted(os.listdir(root), reverse=True):
    if not d.startswith("2025-0"):
        continue
    files = sorted(os.listdir(f"{root}/{d}"))
    for f in files:
        if not bool(re.search(r'[\u4e00-\u9fff]', f)):
            continue
        filepath = f"{root}/{d}/{f}"
        titles = [substr for substr in f.split("_") if bool(re.search(r'[\u4e00-\u9fff]', substr))]
        duration = get_duration(filepath)
        if duration < 8 * 60:
            continue
        approx_num_songs = 2
        if len(titles) < approx_num_songs:
            mins, secs = int(duration // 60), int(duration % 60)
            print(f"{filepath} - {mins:02d}:{secs:02d}")

In [83]:
lyrics = ""
for filepath in tqdm(split_to_limit(f"{root}/2025-01-04/SQ-ST234_爱使我们勇敢+我们爱.mp3")):
    print(filepath)
    audio_file = open(filepath, "rb")
    transcription = client.audio.transcriptions.create(
        # model="gpt-4o-transcribe", 
        model="whisper-1",  
        file=audio_file,
        language="zh",
    )
    lyrics += transcription.text
len(lyrics), lyrics

  0%|          | 0/1 [00:00<?, ?it/s]

/mnt/NextcloudSacmData/sacm.av/files/Recordings/2025-01-04/SQ-ST234_爱使我们勇敢+我们爱.mp3


100%|██████████| 1/1 [00:20<00:00, 20.11s/it]


(607,
 '游刃有余 詞曲 李宗盛 萬軍也何況 你的去所何等何來 我羨慕歌聲你的言語 我心上柔體向你呼籲 萬軍也何況 你的去所何等何來 我羨慕歌聲你的言語 我心上柔體向你呼籲 待我進入你的同在 我不滿足只停留現在 待我進入你的同在 我心刻骨你永恆的愛 待我進入你的同在 我不滿足只停留現在 待我進入你的同在 我心刻骨你永恆的愛 願你的榮耀從天降下來 燃燒每個愛慕你的心 願你的榮耀從天降下來 長伴著你與我們相遇 祂的同在是不能跟其他東西比的 主主幫助我們選擇上好的福分 不只是在我們生活裡選擇耶穌 但在教會的時候 主主幫助我們進入祂的同在 進入祂的榮耀 讓我們 倚靠神為教會 祂的榮耀在我們身上 當其他人看到我們 只要是一個教會 他們就能看到分別 主啊 我們進入祢的同在 因為一個你我們什麼都不能做 幫助我們進入祢的同在 帶我進入 祢的同在 我不滿足 只停留現在 帶我進入 祢的同在 我心刻骨 我心刻骨 祢永恆的愛 帶我進入 祢的同在 我不滿足 只停留現在 帶我進入 祢的同在 我心刻骨 我心刻骨 祢永恆的愛 帶我進入 祢的同在 我不滿足 只停留現在 帶我進入 祢的同在 我心刻骨 祢永恆的愛 願祢的榮耀 從天降下來 燃燒每個愛慕你的心 願祢的榮耀 從天降下來 重返這裡 與我們相遇 願祢的榮耀 從天降下來 燃燒每個愛慕你的心 願祢的榮耀 從天降下來 重返這裡 與我們相遇 我們低頭禱告 由 Amara.org 社群提供的字幕')

In [84]:
titles = {}
title_to_last_chunk_idx = {}

query_lyrics = re.sub(r"[，。！、\n]", " ", lyrics)

chunk_size = 120
for i, start in enumerate(range(0, len(query_lyrics), chunk_size)):
    chunk = query_lyrics[start:start + chunk_size]
    if len(chunk) < 50:
        continue
    query_py = pinyin(chunk)
    scores = [best_window_score(query_py, lyric_py, size=min(len(chunk), 100), step=5) for lyric_py in df.pinyin]
    best_idx = np.argmax(scores)
    best_title = df.iloc[best_idx]["title"]
    best_score = scores[best_idx]
    print(f"[{start}:{start+chunk_size}] {best_title=}, {best_score=}")
    if best_title not in titles:
        titles[best_title] = best_score
    else:
        boost = (i - title_to_last_chunk_idx.get(best_title, -5)) <= 2
        titles[best_title] = max(titles[best_title], best_score) * (1.2 if boost else 1)
    title_to_last_chunk_idx[best_title] = i

print(f"{titles=}")
final_titles = [title for title, score in titles.items() if score > 0.7]
print(f"Songs: {'_'.join(final_titles)}")

[0:120] best_title='带我进入祢的同在', best_score=0.6286472148541113
[120:240] best_title='带我进入祢的同在', best_score=0.6675358539765319
[240:360] best_title='我要看见', best_score=0.5594771241830065
[360:480] best_title='带我进入祢的同在', best_score=0.6503401360544216
[480:600] best_title='带我进入祢的同在', best_score=0.6605263157894737
titles={'带我进入祢的同在': 1.153501955671447, '我要看见': 0.5594771241830065}
Songs: 带我进入祢的同在


In [82]:
df.loc[df.title.str.contains("我们爱")].iloc[0].lyrics

'众人所望，明亮晨星，披戴着一切荣美辉煌。有温柔有怜悯，天地所有权柄，属于我们爱戴的王。\n高举至高的名，耶稣基督，荣美君王，带着世上权柄，将来我们要与他作王。高唱哈利路亚，耶稣基督是弥赛亚，他是圣洁唯一，荣光显明，我们爱戴的王。'

In [ ]:
chunk = query_lyrics[2640:2760]
print("Query:", chunk)
query_pinyin = pinyin(chunk)
for t in ["宁静谷"]:
    inds = df.loc[df.title == t].index
    for idx in inds:
        print(f"[{idx}] {t}: {df.pinyin[idx]}")
        fuzz_score = best_window_score(query_pinyin, df.pinyin[idx], size=100, step=3)
        print(f"{fuzz_score}")

Query:  我学会了信靠他 依靠他 有一次当我 向一位朋友 倾诉我的挣扎时 他推荐我 他推荐给我一首 藏民之群的歌 叫《宁静谷》 歌词中写道 生活中的仓促 生命里的难处 只愿向他来倾诉 平安祝福在这谷 我觉得这首歌 正好讲述了 那段时期 上帝如何 把
[77] 宁静谷: zai wo xin ling shen chu you yi zuo ning jing gu wo he wo qin ai de zhu zai qi zhong an ran man bu sheng huo zhong de cang cu sheng ming li de nan chu zhi yuan xiang ta lai qing su ping an zhu fu zai zhe gu wo yu wo zhu xiang yue zhi chu chang yang zhe fen ning jing an xiang jiu xiang shi zai tian tang wo yu wo zhu xiang yue zhi chu zhu ling wo guo si yin you gu shi wo xi le zou ren sheng lu
score=0.5467158003484595, fuzz_score=0.5852417302798982, 0.2868419756502333


In [ ]:
def get_titles(filepath):
    print(f"Processing {filepath}")

    cropped_paths = split_to_limit(filepath)
    lyrics = ""
    for filepath in tqdm(cropped_paths):
        audio_file = open(filepath, "rb")
        transcription = client.audio.transcriptions.create(
            # model="gpt-4o-transcribe", 
            model="whisper-1", 
            file=audio_file,
            language="zh",
        )
        lyrics += transcription.text

    if not lyrics:
        return []

    titles = {}
    title_to_last_chunk_idx = {}

    query_lyrics = re.sub(r"[，。！、\n]", " ", lyrics)

    chunk_size = 120
    for i, start in enumerate(range(0, len(query_lyrics), chunk_size)):
        chunk = query_lyrics[start:start + chunk_size]
        if len(chunk) < 50:
            continue
        query_py = pinyin(chunk)
        scores = [best_window_score(query_py, lyric_py, size=min(len(chunk), 100), step=5) for lyric_py in df.pinyin]
        best_idx = np.argmax(scores)
        best_title = df.iloc[best_idx]["title"]
        best_score = scores[best_idx]
        print(f"[{start}:{start+chunk_size}] {best_title=}, {best_score=}")
        if best_title in titles and (i - title_to_last_chunk_idx.get(best_title, -5)) <= 2:
            titles[best_title] = max(titles[best_title], best_score) * 1.2
        else:
            titles[best_title] = best_score
        title_to_last_chunk_idx[best_title] = i

    print(f"{titles=}")
    duration = get_duration(filepath)
    if duration > 3 * 60:
        final_titles = [title for title, score in titles.items() if score > 0.7]
    else:
        best_title = max(titles, key=titles.get)
        final_titles = [best_title] if titles[best_title] > 0.7 else []
    return final_titles

In [ ]:
root = "/mnt/NextcloudSacmData/sacm.av/files/Recordings/Past Events/70th ann. rec"
for f in os.listdir(root):
    final_titles = get_titles(f"{root}/{f}")
    print(f"{d}/{f}: {'_'.join(final_titles)}")

In [ ]:
for d in ["2026-06-27", "2026-06-20", "2026-06-14", "2026-05-07", "2026-04-11", "2026-03-08", "2026-03-07", "2026-02-14", "2026-02-08", "2026-02-07", "2026-02-01", "2026-01-25"]:
    matches = match_zoom_to_sq(d, verbose=False)
    print(d, json.dumps(matches, indent=2, ensure_ascii=False))